# SfM -> Panda3D camera track (simplified, Colab)

Set `video_filename` (cell below) and `keyframe_stride`, run every cell in
order, click 3 points on the table when prompted, adjust mesh placement, then
download the composited video at the end. No exercises, no explanations --
just the pipeline.

In [ ]:
# @title Check GPU and install dependencies
import re
import subprocess

smi_output = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(smi_output)
match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi_output)
if not match:
  raise RuntimeError(
      'No GPU detected via nvidia-smi. Go to Runtime > Change runtime type and '
      'select a GPU, then Runtime > Restart session and run all cells again.')
driver_major, driver_minor = int(match.group(1)), int(match.group(2))
print(f'GPU driver supports CUDA up to {driver_major}.{driver_minor}.')
if driver_major < 12:
  raise RuntimeError(
      f'GPU driver only supports up to CUDA {driver_major}.{driver_minor}, but '
      'pycolmap-cuda12 needs CUDA 12 or newer. Try a different Colab GPU runtime.')

# pycolmap-cuda12 depends on cuda-toolkit<13,>=12 (any CUDA-12.x minor) with no
# lower-bound pin, so an unconstrained install can grab a newer CUDA-12 minor
# than this runtime's driver supports. Only pin when the driver itself is on the
# CUDA-12 family -- a driver already on 13+ is backward-compatible with any 12.x.
cuda_toolkit_spec = f'cuda-toolkit=={driver_major}.{driver_minor}.*' if driver_major == 12 else 'cuda-toolkit'
# !pip uninstall -q -y pycolmap
!pip install -q "{cuda_toolkit_spec}" pycolmap-cuda12 absl-py
!pip install Panda3D
!pip install panda3d-gltf  # lets Panda3D load .glb/.gltf models, including
                              # embedded (e.g. skeletal) animations
import pycolmap
if not pycolmap.has_cuda:
  raise RuntimeError(
      'No CUDA GPU detected. Go to Runtime > Change runtime type and select a GPU, '
      'then Runtime > Restart session and run all cells again.')
print(f'pycolmap.has_cuda = {pycolmap.has_cuda}')

# from google.colab import files

import os

In [ ]:

video_filename = "table.mp4" # @param {type: 'string'}
print(f"Using video: {video_filename}")

In [ ]:
# @title Configure dataset directories
from pathlib import Path

save_dir = './'  # @param {type: 'string'}
capture_name = 'capture1'  # @param {type: 'string'}
root_dir = Path(save_dir, capture_name)
rgb_dir = root_dir / 'rgb'
rgb_raw_dir = root_dir / 'rgb-raw'
colmap_dir = root_dir / 'colmap'
colmap_db_path = colmap_dir / 'database.db'
colmap_out_path = colmap_dir / 'sparse'

colmap_out_path.mkdir(exist_ok=True, parents=True)
rgb_raw_dir.mkdir(exist_ok=True, parents=True)

print(f"""Directories configured:
  root_dir = {root_dir}
  rgb_raw_dir = {rgb_raw_dir}
  rgb_dir = {rgb_dir}
  colmap_dir = {colmap_dir}
""")

# An animated, rigged humanoid (.glb) mesh, downloaded once, for the
# "use_mutant_model" toggle in the mesh-placement calibration cell below.
import subprocess

mutant_model_path = Path('mutant.glb')
subprocess.run(
    ['curl', '-fsSL', '-o', str(mutant_model_path),
     'https://raw.githubusercontent.com/prash-red/cs_academy_vfx/main/data/mutant.glb'],
    check=True)
print(f'Downloaded {mutant_model_path}.')

In [ ]:
# @title Flatten into images: read video info

import cv2

# @markdown Extracts every frame of the video into `rgb_raw_dir`. Lower
# @markdown `max_scale` for faster processing on large videos.
video_path = video_filename
max_scale = 1.0  # @param {type:'number'}

# @markdown Running SfM reconstruction on every single frame would be slow and
# @markdown mostly redundant, since consecutive frames barely differ -- so we
# @markdown only reconstruct from every `keyframe_stride`-th frame instead.
keyframe_stride = 10  # @param {type: 'number'}

cap = cv2.VideoCapture(video_path)
input_fps = cap.get(cv2.CAP_PROP_FPS)
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f'Video has {num_frames} frames at {input_fps:.1f} fps.')


In [ ]:
# @title Flatten into images: extract frames
# @markdown Check this if you want to reprocess the frames.
overwrite = True  # @param {type:'boolean'}

import shutil
import subprocess

raw_frames_exist = any(rgb_raw_dir.glob('*.png'))
if raw_frames_exist and not overwrite:
  print(f'Raw frames already extracted in {rgb_raw_dir}, skipping extraction.')
else:
  tmp_rgb_raw_dir = Path('rgb-raw')
  tmp_rgb_raw_dir.mkdir(exist_ok=True, parents=True)
  out_pattern = str(tmp_rgb_raw_dir / '%06d.png')
  subprocess.run(
      ['ffmpeg', '-y', '-i', video_path,
       '-vf', f'scale=iw*{max_scale}:ih*{max_scale}', out_pattern],
      check=True)
  rgb_raw_dir.mkdir(exist_ok=True, parents=True)
  for image_path in tmp_rgb_raw_dir.glob('*.png'):
    shutil.copy2(image_path, rgb_raw_dir / image_path.name)

all_frame_paths = sorted(rgb_raw_dir.glob('*.png'))
keyframe_paths = all_frame_paths[::keyframe_stride]
print(f'Extracted {len(all_frame_paths)} frames; using every '
      f'{keyframe_stride} as a keyframe for reconstruction '
      f'({len(keyframe_paths)} keyframes).')

(root_dir / 'video_fps.txt').write_text(str(input_fps))


In [ ]:
# @title Resize images into different scales.
# @markdown Here we save the keyframes at various resolutions (downsample by a factor of 1, 2, 4, 8). We use area relation interpolation to prevent moire artifacts.
import concurrent.futures
import numpy as np
import cv2
from PIL import Image


def save_image(path, image: np.ndarray) -> None:
  print(f'Saving {path}')
  if not path.parent.exists():
    path.parent.mkdir(exist_ok=True, parents=True)
  with path.open('wb') as f:
    image = Image.fromarray(np.asarray(image))
    image.save(f, format=path.suffix.lstrip('.'))


def image_to_uint8(image: np.ndarray) -> np.ndarray:
  """Convert the image to a uint8 array."""
  if image.dtype == np.uint8:
    return image
  if not issubclass(image.dtype.type, np.floating):
    raise ValueError(
        f'Input image should be a floating type but is of type {image.dtype!r}')
  return (image * 255).clip(0.0, 255).astype(np.uint8)


def make_divisible(image: np.ndarray, divisor: int) -> np.ndarray:
  """Trim the image if not divisible by the divisor."""
  height, width = image.shape[:2]
  if height % divisor == 0 and width % divisor == 0:
    return image

  new_height = height - height % divisor
  new_width = width - width % divisor

  return image[:new_height, :new_width]


def downsample_image(image: np.ndarray, scale: int) -> np.ndarray:
  """Downsamples the image by an integer factor to prevent artifacts."""
  if scale == 1:
    return image

  height, width = image.shape[:2]
  if height % scale > 0 or width % scale > 0:
    raise ValueError(f'Image shape ({height},{width}) must be divisible by the'
                     f' scale ({scale}).')
  out_height, out_width = height // scale, width // scale
  resized = cv2.resize(image, (out_width, out_height), cv2.INTER_AREA)
  return resized



image_scales = "1,2,4,8"  # @param {type: "string"}
image_scales = [int(x) for x in image_scales.split(',')]

tmp_rgb_dir = Path('rgb')

resized_already = (rgb_dir / '1x').exists() and not overwrite
if resized_already:
  print(f'RGB frames already resized in {rgb_dir}, skipping resize.')
else:
  for image_path in keyframe_paths:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = make_divisible(image, max(image_scales))
    for scale in image_scales:
      save_image(
          tmp_rgb_dir / f'{scale}x/{image_path.stem}.png',
          image_to_uint8(downsample_image(image, scale)))

  for scale in image_scales:
    scale_src_dir = tmp_rgb_dir / f'{scale}x'
    scale_dst_dir = rgb_dir / f'{scale}x'
    scale_dst_dir.mkdir(exist_ok=True, parents=True)
    for image_path in scale_src_dir.glob('*.png'):
      shutil.copy2(image_path, scale_dst_dir / image_path.name)


In [ ]:
# @title Show example frame, then extract & match features.
# @markdown Make sure that the video was processed correctly.
# @markdown If this gives an exception, try running the preceding cell one more time--sometimes uploading to Google Drive can fail.

from pathlib import Path
from PIL import Image

image_paths = list((rgb_dir / '1x').iterdir())
Image.open(image_paths[0])

import pycolmap

share_intrinsics = True  # @param {type: 'boolean'}
assume_upright_cameras = True  # @param {type: 'boolean'}
colmap_image_scale = 2  # @param {type: 'number'}
colmap_rgb_dir = rgb_dir / f'{colmap_image_scale}x'

# @markdown Requires a CUDA-enabled GPU runtime (on Colab: Runtime > Change runtime
# @markdown type > GPU). Fails loudly instead of silently falling back to CPU, since
# @markdown a silent CPU fallback would make SIFT extraction/matching far slower.
use_gpu = True  # @param {type: 'boolean'}
if use_gpu and not pycolmap.has_cuda:
  raise RuntimeError(
      'use_gpu is set but this pycolmap build has no CUDA support, or no GPU is '
      'attached to this runtime. On Colab: Runtime > Change runtime type > GPU. '
      'Otherwise set use_gpu = False to run on CPU.')
device = pycolmap.Device.cuda if use_gpu else pycolmap.Device.cpu
print(f'pycolmap.has_cuda = {pycolmap.has_cuda}; using device = {device.name}')

if colmap_db_path.exists():
  colmap_db_path.unlink()

# Configure reader options
reader_opts = pycolmap.ImageReaderOptions()
if hasattr(reader_opts, 'single_camera'):
    reader_opts.single_camera = share_intrinsics

# In pycolmap 4.x, SiftExtractionOptions must be wrapped inside FeatureExtractionOptions
sift_extraction_opts = pycolmap.SiftExtractionOptions()
if hasattr(sift_extraction_opts, 'use_gpu'):
    sift_extraction_opts.use_gpu = use_gpu
sift_extraction_opts.upright = assume_upright_cameras

extraction_opts = pycolmap.FeatureExtractionOptions()
extraction_opts.sift = sift_extraction_opts

print(f"Extracting features from: {colmap_rgb_dir}")
pycolmap.extract_features(
    database_path=str(colmap_db_path),
    image_path=str(colmap_rgb_dir),
    reader_options=reader_opts,
    extraction_options=extraction_opts,
    device=device
)

# Configure SIFT matching options
sift_matching_options = pycolmap.SiftMatchingOptions()
if hasattr(sift_matching_options, 'use_gpu'):
    sift_matching_options.use_gpu = use_gpu

# In pycolmap 4.x, SiftMatchingOptions must be wrapped inside FeatureMatchingOptions
matching_opts = pycolmap.FeatureMatchingOptions()
matching_opts.sift = sift_matching_options

print("Matching features exhaustively...")
pycolmap.match_exhaustive(
    database_path=str(colmap_db_path),
    matching_options=matching_opts,
    device=device
)

In [ ]:
# @title Reconstruction.
# @markdown Run structure-from-motion to compute camera parameters using pycolmap.

refine_principal_point = True  #@param {type:"boolean"}
min_num_matches = 10 # @param {type: 'number'}
filter_max_reproj_error = 2  # @param {type: 'number'}
tri_complete_max_reproj_error = 2  # @param {type: 'number'}

# Configure the low-level Incremental Mapper options
mapper_options = pycolmap.IncrementalMapperOptions()
if hasattr(mapper_options, 'min_num_matches'):
    mapper_options.min_num_matches = int(min_num_matches)
if hasattr(mapper_options, 'filter_max_reproj_error'):
    mapper_options.filter_max_reproj_error = float(filter_max_reproj_error)
if hasattr(mapper_options, 'tri_complete_max_reproj_error'):
    mapper_options.tri_complete_max_reproj_error = float(tri_complete_max_reproj_error)
if hasattr(mapper_options, 'ba_refine_principal_point'):
    mapper_options.ba_refine_principal_point = refine_principal_point

# In pycolmap 4.x, incremental_mapping expects IncrementalPipelineOptions
pipeline_options = pycolmap.IncrementalPipelineOptions()
pipeline_options.mapper = mapper_options

# Ensure output directory exists
colmap_out_path.mkdir(parents=True, exist_ok=True)

print(f"Running incremental mapping on: {colmap_db_path}")
maps = pycolmap.incremental_mapping(
    database_path=str(colmap_db_path),
    image_path=str(colmap_rgb_dir),
    output_path=str(colmap_out_path),
    options=pipeline_options
)

if maps:
    print(f"Reconstruction successful! Found {len(maps)} sub-models.")
else:
    print("Reconstruction failed to produce any maps.")

In [ ]:
# @title Verify that SfM worked.

def _has_model(model_dir):
  return (model_dir / 'cameras.bin').exists() or (model_dir / 'cameras.txt').exists()

if not colmap_db_path.exists():
  raise RuntimeError(f'The SfM database does not exist, did you run the reconstruction?')
# pycolmap incremental_mapping writes models to numbered folders (0, 1, etc.) --
# there can be more than one if the images didn't all connect into one component.
elif not any(_has_model(d) for d in colmap_out_path.iterdir() if d.is_dir()):
  raise RuntimeError("""
SfM seems to have failed to save the model. Try some of the following options:
 - Decrease `keyframe_stride` when flattening to images, so more keyframes
   get reconstructed. There should be at least 50-ish keyframes.
 - Decrease `min_num_matches`.
 - If your images aren't upright, uncheck `assume_upright_cameras`.
""")
else:
  print("Everything looks good! Found sparse reconstruction model(s).")


In [ ]:
# @title Define Camera class.
import copy
import json
from typing import Optional, Tuple, Union

import numpy as np


class Camera:
  """Minimal camera model used by this notebook."""

  def __init__(self,
               orientation: np.ndarray,
               position: np.ndarray,
               focal_length: Union[np.ndarray, float],
               principal_point: np.ndarray,
               image_size: np.ndarray,
               skew: Union[np.ndarray, float] = 0.0,
               pixel_aspect_ratio: Union[np.ndarray, float] = 1.0,
               radial_distortion: Optional[np.ndarray] = None,
               tangential_distortion: Optional[np.ndarray] = None,
               dtype=np.float32):
    if radial_distortion is None:
      radial_distortion = np.array([0.0, 0.0, 0.0], dtype=dtype)
    if tangential_distortion is None:
      tangential_distortion = np.array([0.0, 0.0], dtype=dtype)

    self.orientation = np.array(orientation, dtype=dtype)
    self.position = np.array(position, dtype=dtype)
    self.focal_length = np.array(focal_length, dtype=dtype)
    self.principal_point = np.array(principal_point, dtype=dtype)
    self.skew = np.array(skew, dtype=dtype)
    self.pixel_aspect_ratio = np.array(pixel_aspect_ratio, dtype=dtype)
    self.radial_distortion = np.array(radial_distortion, dtype=dtype)
    self.tangential_distortion = np.array(tangential_distortion, dtype=dtype)
    self.image_size = np.array(image_size, np.uint32)
    self.dtype = dtype

  @classmethod
  def from_json(cls, path):
    with open(path, 'r') as fp:
      camera_json = json.load(fp)
    if 'tangential' in camera_json:
      camera_json['tangential_distortion'] = camera_json['tangential']
    return cls(
        orientation=np.asarray(camera_json['orientation']),
        position=np.asarray(camera_json['position']),
        focal_length=camera_json['focal_length'],
        principal_point=np.asarray(camera_json['principal_point']),
        skew=camera_json['skew'],
        pixel_aspect_ratio=camera_json['pixel_aspect_ratio'],
        radial_distortion=np.asarray(camera_json['radial_distortion']),
        tangential_distortion=np.asarray(camera_json['tangential_distortion']),
        image_size=np.asarray(camera_json['image_size']),
    )

  def to_json(self):
    return {
        k: (v.tolist() if hasattr(v, 'tolist') else v)
        for k, v in self.get_parameters().items()
    }

  def get_parameters(self):
    return {
        'orientation': self.orientation,
        'position': self.position,
        'focal_length': self.focal_length,
        'principal_point': self.principal_point,
        'skew': self.skew,
        'pixel_aspect_ratio': self.pixel_aspect_ratio,
        'radial_distortion': self.radial_distortion,
        'tangential_distortion': self.tangential_distortion,
        'image_size': self.image_size,
    }

  @property
  def image_size_x(self):
    return self.image_size[0]

  @property
  def image_size_y(self):
    return self.image_size[1]

  @property
  def principal_point_x(self):
    return self.principal_point[0]

  @property
  def principal_point_y(self):
    return self.principal_point[1]

  @property
  def scale_factor_x(self):
    return self.focal_length

  @property
  def scale_factor_y(self):
    return self.focal_length * self.pixel_aspect_ratio

  @property
  def optical_axis(self):
    return self.orientation[2, :]

  @property
  def translation(self):
    return -np.matmul(self.orientation, self.position)

  @property
  def has_radial_distortion(self):
    return any(self.radial_distortion != 0.0)

  @property
  def has_tangential_distortion(self):
    return any(self.tangential_distortion != 0.0)

  def copy(self):
    return copy.deepcopy(self)

  def pixel_to_local_rays(self, pixels: np.ndarray):
    pixels = np.asarray(pixels, dtype=self.dtype)
    y = ((pixels[..., 1] - self.principal_point_y) / self.scale_factor_y)
    x = ((pixels[..., 0] - self.principal_point_x - y * self.skew) /
         self.scale_factor_x)

    dirs = np.stack([x, y, np.ones_like(x)], axis=-1)
    return dirs / np.linalg.norm(dirs, axis=-1, keepdims=True)

  def pixels_to_rays(self, pixels: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if np.asarray(pixels).shape[-1] != 2:
      raise ValueError('The last dimension of pixels must be 2.')
    pixels = np.asarray(pixels, dtype=self.dtype)
    batch_shape = pixels.shape[:-1]
    pixels = np.reshape(pixels, (-1, 2))
    local_rays_dir = self.pixel_to_local_rays(pixels)
    rays_dir = np.matmul(self.orientation.T, local_rays_dir[..., np.newaxis])
    rays_dir = np.squeeze(rays_dir, axis=-1)
    rays_dir /= np.linalg.norm(rays_dir, axis=-1, keepdims=True)
    return rays_dir.reshape((*batch_shape, 3))

  def pixels_to_points(self, pixels: np.ndarray, depth: np.ndarray):
    rays_through_pixels = self.pixels_to_rays(pixels)
    cosa = np.matmul(rays_through_pixels, self.optical_axis)
    return (rays_through_pixels * depth[..., np.newaxis] / cosa[..., np.newaxis] +
            self.position)

  def points_to_local_points(self, points: np.ndarray):
    translated_points = np.asarray(points, dtype=self.dtype) - self.position
    return (np.matmul(self.orientation, translated_points.T)).T

  def project(self, points: np.ndarray):
    points = np.asarray(points, dtype=self.dtype).reshape((-1, 3))
    local_points = self.points_to_local_points(points)
    x = local_points[..., 0] / local_points[..., 2]
    y = local_points[..., 1] / local_points[..., 2]
    pixel_x = self.focal_length * x + self.skew * y + self.principal_point_x
    pixel_y = (self.focal_length * self.pixel_aspect_ratio * y +
               self.principal_point_y)
    return np.stack([pixel_x, pixel_y], axis=-1)

  def get_pixel_centers(self):
    xx, yy = np.meshgrid(np.arange(self.image_size_x, dtype=self.dtype),
                         np.arange(self.image_size_y, dtype=self.dtype))
    return np.stack([xx, yy], axis=-1) + 0.5

  def scale(self, scale: float):
    if scale <= 0:
      raise ValueError('scale needs to be positive.')
    return Camera(
        orientation=self.orientation.copy(),
        position=self.position.copy(),
        focal_length=self.focal_length * scale,
        principal_point=self.principal_point.copy() * scale,
        skew=self.skew,
        pixel_aspect_ratio=self.pixel_aspect_ratio,
        radial_distortion=self.radial_distortion.copy(),
        tangential_distortion=self.tangential_distortion.copy(),
        image_size=np.array((int(round(self.image_size[0] * scale)),
                             int(round(self.image_size[1] * scale)))),
        dtype=self.dtype)

  def look_at(self, position, look_at, up, eps=1e-6):
    look_at_camera = self.copy()
    optical_axis = np.asarray(look_at) - np.asarray(position)
    norm = np.linalg.norm(optical_axis)
    if norm < eps:
      raise ValueError('The camera center and look at position are too close.')
    optical_axis /= norm
    right_vector = np.cross(optical_axis, up)
    norm = np.linalg.norm(right_vector)
    if norm < eps:
      raise ValueError('The up-vector is parallel to the optical axis.')
    right_vector /= norm
    camera_rotation = np.identity(3, dtype=self.dtype)
    camera_rotation[0, :] = right_vector
    camera_rotation[1, :] = np.cross(optical_axis, right_vector)
    camera_rotation[2, :] = optical_axis
    look_at_camera.position = np.asarray(position, dtype=self.dtype)
    look_at_camera.orientation = camera_rotation
    return look_at_camera

In [ ]:
# @title Define Scene Manager.
from absl import logging
from typing import Dict
from pathlib import Path
import numpy as np
import pycolmap
import cv2


def convert_colmap_camera(colmap_camera, colmap_image):
  """Converts a pycolmap image to an SFM camera."""
  pose = colmap_image.cam_from_world
  if callable(pose):
    pose = pose()
  camera_rotation = pose.rotation.matrix()
  camera_translation = pose.translation
  camera_position = -(camera_rotation.T @ camera_translation)

  params = np.asarray(colmap_camera.params)
  focal_length = params[0]
  cx, cy = params[1], params[2]
  radial = np.array([params[3] if len(params) > 3 else 0.0,
                     params[4] if len(params) > 4 else 0.0,
                     params[5] if len(params) > 5 else 0.0])
  tangential = np.array([params[6] if len(params) > 6 else 0.0,
                         params[7] if len(params) > 7 else 0.0])

  return Camera(
      orientation=camera_rotation,
      position=camera_position,
      focal_length=focal_length,
      pixel_aspect_ratio=1.0,
      principal_point=np.array([cx, cy]),
      radial_distortion=radial,
      tangential_distortion=tangential,
      skew=0.0,
      image_size=np.array([colmap_camera.width, colmap_camera.height]))


def filter_outlier_points(points, inner_percentile):
  """Filters outlier points."""
  if points is None or len(points) == 0:
    return points
  outer = 1.0 - inner_percentile
  lower = outer / 2.0
  upper = 1.0 - lower
  centers_min = np.quantile(points, lower, axis=0)
  centers_max = np.quantile(points, upper, axis=0)
  result = points.copy()
  too_near = np.any(result < centers_min[None, :], axis=1)
  too_far = np.any(result > centers_max[None, :], axis=1)
  return result[~(too_near | too_far)]


def reprojection_error(points, pixels, camera):
  """Computes reprojection error per-point for a single camera."""
  projected = camera.project(points)
  return np.linalg.norm(projected - pixels, axis=-1)


def average_reprojection_errors(points, pixels, cameras):
  """Computes the average reprojection errors of the points."""
  cam_errors = []
  for i, camera in enumerate(cameras):
    cam_error = reprojection_error(points, pixels[:, i], camera)
    cam_errors.append(cam_error)
  cam_error = np.stack(cam_errors)
  return cam_error.mean(axis=1)


def _get_camera_translation(camera):
  """Computes the extrinsic translation of the camera."""
  rot_mat = camera.orientation
  return -camera.position.dot(rot_mat.T)


def _transform_camera(camera, transform_mat):
  """Transforms the camera using the given transformation matrix."""
  if transform_mat.shape != (3, 4):
    raise ValueError('transform_mat should be a 3x4 transformation matrix.')
  rotation = camera.orientation @ transform_mat[:, :3].T
  position = camera.position @ transform_mat[:, :3].T + transform_mat[:, 3]
  new_camera = camera.copy()
  new_camera.orientation = rotation
  new_camera.position = position
  return new_camera


def triangulate_pixels(pixels, cameras):
  """Triangulates one 3D point from one pixel per camera."""
  if pixels.shape != (len(cameras), 2):
    raise ValueError(
        f'The number of pixels ({len(pixels)}) must be equal to the number '
        f'of cameras ({len(cameras)}).')
  eye = np.eye(3, dtype=np.float64)
  a_mat = np.zeros((3, 3), dtype=np.float64)
  b_vec = np.zeros(3, dtype=np.float64)
  for pixel, camera in zip(pixels, cameras):
    origin = np.asarray(camera.position, dtype=np.float64)
    direction = np.asarray(camera.pixels_to_rays(np.asarray(pixel, dtype=camera.dtype)[None, :]), dtype=np.float64)[0]
    proj = eye - np.outer(direction, direction)
    a_mat += proj
    b_vec += proj @ origin
  try:
    return np.linalg.solve(a_mat, b_vec)
  except np.linalg.LinAlgError:
    return np.linalg.lstsq(a_mat, b_vec, rcond=None)[0]


def _pycolmap_to_sfm_cameras(reconstruction: pycolmap.Reconstruction) -> Dict[str, Camera]:
  """Creates SFM cameras keyed by the original image stem."""
  sfm_cameras = {}
  for image_id, image in reconstruction.images.items():

    camera = reconstruction.cameras[image.camera_id]
    image_key = Path(image.name).stem
    sfm_cameras[image_key] = convert_colmap_camera(camera, image)
  return sfm_cameras


class SceneManager:
  """A thin wrapper around pycolmap Reconstruction."""

  @classmethod
  def from_pycolmap(cls, colmap_path, image_path, min_track_length=10):
    """Create a scene manager using pycolmap."""
    reconstruction = pycolmap.Reconstruction(str(colmap_path))
    sfm_cameras = _pycolmap_to_sfm_cameras(reconstruction)
    points = []
    colors = []
    for point3D_id, point3D in reconstruction.points3D.items():
      if point3D.track.length() >= min_track_length:
        points.append(point3D.xyz)
        colors.append(point3D.color)
    return cls(sfm_cameras,
                np.array(points) if points else np.empty((0, 3)),
                image_path,
                colors=np.array(colors) if colors else np.empty((0, 3)))

  def __init__(self, cameras, points, image_path, colors=None):
    self.image_path = Path(image_path)
    self.camera_dict = cameras
    self.points = points
    self.colors = colors if colors is not None else np.zeros((len(points), 3))
    logging.info('Created scene manager with %d cameras', len(self.camera_dict))

  def __len__(self):
    return len(self.camera_dict)

  @property
  def image_ids(self):
    return sorted(self.camera_dict.keys())

  @property
  def camera_list(self):
    return [self.camera_dict[i] for i in self.image_ids]

  @property
  def camera_positions(self):
    """Returns an array of camera positions."""
    return np.stack([camera.position for camera in self.camera_list]) if self.camera_dict else np.empty((0, 3))

  def load_image(self, image_id):
    """Loads the image with the specified image_id."""
    path = self.image_path / f'{image_id}.png'
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
      raise FileNotFoundError(f'Could not read image at {path}')
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

  def triangulate_pixels(self, pixels):
    """Triangulates the pixels across all cameras in the scene."""
    if pixels.shape != (len(self), 2):
      raise ValueError(
          f'The number of pixels ({len(pixels)}) must be equal to the number '
          f'of cameras ({len(self)}).')
    return triangulate_pixels(pixels, self.camera_list)

  def change_basis(self, axes, center):
    """Change the basis of the scene."""
    transform_mat = np.zeros((3, 4))
    transform_mat[:3, :3] = axes.T
    transform_mat[:, 3] = -(center @ axes)
    return self.transform(transform_mat)

  def transform(self, transform_mat):
    """Transform the scene using a transformation matrix."""
    if transform_mat.shape != (3, 4):
      raise ValueError('transform_mat should be a 3x4 transformation matrix.')

    points = None
    if self.points is not None:
      points = self.points.copy()
      points = points @ transform_mat[:, :3].T + transform_mat[:, 3]

    new_cameras = {}
    for image_id, camera in self.camera_dict.items():
      new_cameras[image_id] = _transform_camera(camera, transform_mat)

    return SceneManager(new_cameras, points, self.image_path, colors=self.colors)

  def filter_images(self, image_ids):
    num_filtered = 0
    for image_id in image_ids:
      if self.camera_dict.pop(image_id, None) is not None:
        num_filtered += 1
    return num_filtered

In [ ]:
# @title Load pycolmap scene.
import plotly.graph_objs as go
from pathlib import Path

# incremental_mapping can split the reconstruction across multiple numbered
# folders if not all images connected into one component; pick the one with
# the most registered images rather than assuming it's always folder '0'.
best_model_dir, best_num_images = None, -1
for model_dir in sorted((colmap_dir / 'sparse').iterdir()):
  if not model_dir.is_dir() or not _has_model(model_dir):
    continue
  num_images = len(pycolmap.Reconstruction(str(model_dir)).images)
  print(f'  Model {model_dir.name}: {num_images} registered images.')
  if num_images > best_num_images:
    best_model_dir, best_num_images = model_dir, num_images
print(f'Selected model {best_model_dir.name} with {best_num_images} registered images.')

scene_manager = SceneManager.from_pycolmap(
    best_model_dir,
    rgb_dir / f'1x',
    min_track_length=5)

if colmap_image_scale > 1:
  print(f'Scaling COLMAP cameras back to 1x from {colmap_image_scale}x.')
  for item_id in scene_manager.image_ids:
    camera = scene_manager.camera_dict[item_id]
    scene_manager.camera_dict[item_id] = camera.scale(colmap_image_scale)

point_colors = [f'rgb({r},{g},{b})' for r, g, b in scene_manager.colors]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=scene_manager.points[:, 0],
    y=scene_manager.points[:, 1],
    z=scene_manager.points[:, 2],
    mode='markers',
    marker=dict(size=2, color=point_colors),
    name='Points',
))
fig.add_trace(go.Scatter3d(
    x=scene_manager.camera_positions[:, 0],
    y=scene_manager.camera_positions[:, 1],
    z=scene_manager.camera_positions[:, 2],
    mode='markers',
    marker=dict(size=5, color='red'),
    name='Cameras',
))
fig.update_layout(scene_dragmode='orbit', title='Sparse Reconstruction')
fig.show()

In [ ]:
# @title Normalize scene based on landmarks.
# @markdown This workshop doesn't use facial-landmark-based normalization
# @markdown (that's for face-capture pipelines) -- the bounding-box-based
# @markdown centering/scaling below is what actually normalizes the scene
# @markdown for table/object captures. This cell is a no-op passthrough.
new_scene_manager = scene_manager


In [ ]:
# @title Compute the scene bounding box

def get_bbox_corners(points):
  lower = points.min(axis=0)
  upper = points.max(axis=0)
  return np.stack([lower, upper])


points = filter_outlier_points(new_scene_manager.points, 0.95)
bbox_corners = get_bbox_corners(
    np.concatenate([points, new_scene_manager.camera_positions], axis=0))

print(f'Bounding box: {bbox_corners[0]} to {bbox_corners[1]}')


In [ ]:
# @title Compute scene center and scale
def compute_scene_center_and_scale(bbox_corners):
  """Returns the midpoint between the two bbox corners (the center of the
  scene), and 1 / (the distance between them) as a scale that maps the
  scene into a roughly unit-sized cube."""
  center = (bbox_corners[0] + bbox_corners[1]) / 2
  diagonal_length = np.linalg.norm(bbox_corners[1] - bbox_corners[0])
  scale = 1.0 / diagonal_length
  return center, scale


scene_center, scene_scale = compute_scene_center_and_scale(bbox_corners)
print(f'Scene Center: {scene_center}')
print(f'Scene Scale: {scene_scale}')

In [ ]:
# @title Export camera track for Panda3D compositing.
# @markdown Writes a camera track JSON in the shared schema used by
# @markdown composite_track.py (same schema as dust3r_track_extraction.py /
# @markdown opencv_track_extraction.py):
# @markdown   [{"position": [x,y,z], "forward": [x,y,z], "up": [x,y,z], "focal_px": f}, ...]
# @markdown one entry per raw video frame, in CV convention (X-right, Y-down, Z-forward).
# @markdown Camera.orientation's rows are the camera's local axes expressed in world
# @markdown coordinates, so row 2 (Z) is "forward" and -row 1 (-Y) is "up".

from scipy.spatial.transform import Rotation, Slerp


def interpolate_camera(cam_a, cam_b, t):
  """Interpolates pose between two cameras; t=0 -> cam_a, t=1 -> cam_b."""
  position = (1 - t) * cam_a.position + t * cam_b.position
  focal_length = (1 - t) * cam_a.focal_length + t * cam_b.focal_length
  rotations = Rotation.from_matrix(np.stack([cam_a.orientation, cam_b.orientation]))
  orientation = Slerp([0, 1], rotations)([t])[0].as_matrix()
  interpolated = cam_a.copy()
  interpolated.position = position.astype(cam_a.dtype)
  interpolated.orientation = orientation.astype(cam_a.dtype)
  interpolated.focal_length = np.array(focal_length, dtype=cam_a.dtype)
  return interpolated


frame_stems = sorted(p.stem for p in rgb_raw_dir.glob('*.png'))
if not frame_stems:
  raise RuntimeError(f'No raw frames found in {rgb_raw_dir}.')

cameras_by_frame = [new_scene_manager.camera_dict.get(stem) for stem in frame_stems]
registered_indices = [i for i, cam in enumerate(cameras_by_frame) if cam is not None]
if not registered_indices:
  raise RuntimeError('No frames were registered by SfM; cannot export a camera track.')

# Look up bracketing neighbors among the *originally* registered frames only, so
# a run of consecutive gaps interpolates against real SfM poses on both sides
# rather than compounding error against already-filled-in neighbors.
for i, cam in enumerate(cameras_by_frame):
  if cam is not None:
    continue
  before = max((j for j in registered_indices if j < i), default=None)
  after = min((j for j in registered_indices if j > i), default=None)
  if before is not None and after is not None:
    t = (i - before) / (after - before)
    cameras_by_frame[i] = interpolate_camera(
        cameras_by_frame[before], cameras_by_frame[after], t)
    print(f'Frame {frame_stems[i]} was not registered by SfM; interpolated '
          f'between frames {frame_stems[before]} and {frame_stems[after]}.')
  else:
    # Leading/trailing gap: no bracketing frame on one side, so hold the
    # nearest registered pose instead of extrapolating.
    nearest = before if before is not None else after
    cameras_by_frame[i] = cameras_by_frame[nearest]
    print(f'Frame {frame_stems[i]} was not registered by SfM; '
          f'reusing pose from frame {frame_stems[nearest]} (edge of track).')

camera_track = []
for cam in cameras_by_frame:
  forward = cam.orientation[2, :]
  up = -cam.orientation[1, :]
  camera_track.append({
      'position': cam.position.tolist(),
      'forward': forward.tolist(),
      'up': up.tolist(),
      'focal_px': float(cam.focal_length),
  })

camera_track_path = root_dir / 'camera_track.json'
with camera_track_path.open('w') as f:
  json.dump(camera_track, f, indent=2)

print(f'Wrote camera track with {len(camera_track)} frames '
      f'({len(registered_indices)} registered by SfM) to {camera_track_path}')

sparse_points_path = root_dir / 'sparse_points.npy'
np.save(sparse_points_path, np.asarray(new_scene_manager.points, dtype=np.float64))
print(f'Wrote {len(new_scene_manager.points)} sparse points to {sparse_points_path}')


In [ ]:
# @title Composite: install Panda3D and resume from disk.
# @markdown Installs Panda3D (skipped if already installed, e.g. in this
# @markdown project's `.venv-panda3d`) and re-establishes everything the
# @markdown compositing cells below need, purely from files already written
# @markdown to disk by a previous run of the cells above -- so after
# @markdown restarting the kernel you can run from THIS cell onward without
# @markdown re-running SfM/reconstruction. Requires that `camera_track.json`
# @markdown (and `video_fps.txt`, written during extraction) already exist
# @markdown under `root_dir` from an earlier session.

import panda3d

import json
import subprocess
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

# @markdown Must match whatever these were set to in "Configure dataset
# @markdown directories" when SfM was originally run.
save_dir = './'  # @param {type: 'string'}
capture_name = 'capture1'  # @param {type: 'string'}
root_dir = Path(save_dir, capture_name)
rgb_dir = root_dir / 'rgb'
rgb_raw_dir = root_dir / 'rgb-raw'

if not (root_dir / 'camera_track.json').exists():
  raise RuntimeError(
      f'{root_dir / "camera_track.json"} not found -- run the SfM cells '
      'above at least once (for this capture_name) before resuming here.')

sparse_points_path = root_dir / 'sparse_points.npy'
if not sparse_points_path.exists():
  raise RuntimeError(
      f'{sparse_points_path} not found -- run the SfM cells above at least '
      'once (for this capture_name) before resuming here.')
sparse_points_cv = np.load(sparse_points_path)

video_fps_path = root_dir / 'video_fps.txt'
if video_fps_path.exists():
  input_fps = float(video_fps_path.read_text().strip())
else:
  # Fell over to a capture from before video_fps.txt was written -- input_fps
  # must have already run through "Flatten into images" in this same kernel
  # session for the composite cells below to have a value at all.
  if 'input_fps' not in globals():
    raise RuntimeError(
        f'{video_fps_path} not found and `input_fps` is not set. Either '
        "re-run \"Flatten into images\" once to write it, or set `input_fps` "
        "manually to your source video's native frame rate.")

print(f'Resuming capture "{capture_name}" from {root_dir} (input_fps={input_fps}).')


In [ ]:
# @title Composite: define mesh renderer.
# @markdown Sets up an offscreen Panda3D renderer that draws a 3D mesh with a
# @markdown transparent alpha channel, so it can be alpha-composited onto video
# @markdown frames. Both the mesh and the calibration axes gizmo hang off a
# @markdown "basis" node that can be repositioned/reoriented to any coordinate
# @markdown frame (e.g. the table-plane basis computed below) -- everything
# @markdown placed relative to it (mesh position AND orientation) is then
# @markdown automatically expressed in that frame, using Panda3D's normal
# @markdown parent-relative setPos/setHpr semantics.

from panda3d.core import loadPrcFileData

# Must be set before ShowBase() is created.
loadPrcFileData("", "window-type offscreen")
loadPrcFileData("", "audio-library-name null")
loadPrcFileData("", "framebuffer-alpha true")
loadPrcFileData("", "depth-bits 24")

import numpy as np
from PIL import Image
from direct.showbase.ShowBase import ShowBase
from direct.actor.Actor import Actor
from panda3d.core import (AmbientLight, DirectionalLight, LineSegs, Mat3, Point3,
                           Quat, TextNode, Vec4, Vec3)


def cv_to_panda(v):
  """(x, y, z) in CV convention (X-right, Y-down, Z-forward) -> Panda3D
  convention (X-right, Y-forward, Z-up)."""
  x, y, z = v
  return Vec3(x, z, -y)


def basis_point_to_world_cv(offset_cv, origin_cv, right_cv, forward_cv, up_cv):
  """Converts a point expressed in a CV-convention basis (X=right, Y=down,
  Z=forward, built from the given axes) into absolute world CV coordinates --
  the inverse of what set_basis()/place_mesh() do inside Panda3D, used outside
  Panda3D (e.g. for the scene-scale heuristic) to know where a basis-relative
  point actually sits in the world."""
  ox, oy, oz = offset_cv
  down_cv = -np.asarray(up_cv)
  return (np.asarray(origin_cv) + ox * np.asarray(right_cv) +
          oy * down_cv + oz * np.asarray(forward_cv))


class MeshRenderer(ShowBase):
  def __init__(self, model_path, win_size):
    loadPrcFileData("", f"win-size {win_size[0]} {win_size[1]}")
    super().__init__()

    # Transparent clear color: alpha=0 outside the mesh so compositing onto
    # video only shows the rendered geometry, not a background plate.
    self.win.setClearColor(Vec4(0, 0, 0, 0))

    # The calibration basis: a node positioned/oriented so its own local axes
    # ARE whatever (right, down, forward) triple was last passed to
    # set_basis() -- identity (world CV axes at the world origin) by default.
    # The mesh and the axes gizmo both hang off this node, so their local
    # setPos/setHpr calls are automatically relative to that basis.
    self.basis_node = self.render.attachNewNode("basis_node")

    self.mesh_pivot = self.basis_node.attachNewNode("mesh_pivot")
    self._load_mesh(model_path)

    self.axes_node = None

    ambient = AmbientLight("ambient")
    ambient.setColor(Vec4(0.3, 0.3, 0.3, 1))
    self.render.setLight(self.render.attachNewNode(ambient))

    key_light = DirectionalLight("key")
    key_light.setColor(Vec4(0.9, 0.9, 0.85, 1))
    key_np = self.render.attachNewNode(key_light)
    key_np.setHpr(-30, -45, 0)
    self.render.setLight(key_np)

    fill_light = DirectionalLight("fill")
    fill_light.setColor(Vec4(0.3, 0.3, 0.4, 1))
    fill_np = self.render.attachNewNode(fill_light)
    fill_np.setHpr(150, -20, 0)
    self.render.setLight(fill_np)

  def _load_mesh(self, model_path):
    # Actor (not plain loader.loadModel) so that an embedded animation --
    # e.g. a rigged/skinned .glb humanoid -- can actually be posed and
    # played, not just rendered in its rest pose. Actor works fine for
    # non-animated models too, since it's just a NodePath subclass.
    self.mesh = Actor(model_path or "models/panda-model")
    self.mesh.reparentTo(self.mesh_pivot)

  def reset_mesh(self, model_path):
    """Swaps in a freshly-loaded model, discarding the old one. Lets a
    notebook cell reload the mesh (e.g. after changing `model_path`) without
    spawning a second ShowBase, which Panda3D does not allow per process."""
    self.mesh.removeNode()
    self._load_mesh(model_path)

  def set_anim_time(self, anim_name, t_seconds, loop=True):
    """Poses the mesh's `anim_name` animation clip at time t_seconds since
    the start of the clip (see `mesh.getAnimNames()` for the exact names
    available on this model). Loops by default, so a short clip (e.g. a
    walk cycle) repeats across the whole length of the video."""
    if anim_name not in self.mesh.getAnimNames():
      return
    frame_rate = self.mesh.getFrameRate(anim_name) or 24.0
    num_frames = self.mesh.getNumFrames(anim_name)
    if not num_frames:
      return
    frame = t_seconds * frame_rate
    frame = frame % num_frames if loop else min(frame, num_frames - 1)
    self.mesh.pose(anim_name, int(frame))

  def set_basis(self, origin_cv, right_cv, forward_cv, up_cv):
    """Positions/orients the calibration basis at origin_cv so its own local
    (X, Y, Z) axes are (right_cv, -up_cv, forward_cv) -- i.e. a CV-convention
    (X=right, Y=down, Z=forward) frame built from the given axes."""
    self.basis_node.setPos(cv_to_panda(origin_cv))
    right_p = cv_to_panda(right_cv)
    forward_p = cv_to_panda(forward_cv)
    up_p = cv_to_panda(up_cv)
    basis_mat = Mat3(right_p.x, right_p.y, right_p.z,
                      forward_p.x, forward_p.y, forward_p.z,
                      up_p.x, up_p.y, up_p.z)
    quat = Quat()
    quat.setFromMatrix(basis_mat)
    self.basis_node.setQuat(quat)

  def place_mesh(self, offset_cv, size_fraction, scene_scale, hpr=(0.0, 0.0, 0.0)):
    """Centers the mesh on itself, scales it to `size_fraction` of the
    track's characteristic scale, then moves the pivot to `offset_cv` (in the
    current basis) with the given orientation (heading, pitch, roll, also
    relative to the current basis)."""
    self.mesh_pivot.show()
    bounds = self.mesh.getTightBounds()
    if bounds is not None:
      min_point, max_point = bounds
      center = (min_point + max_point) * 0.5
      size = max_point - min_point
      max_dim = max(size.x, size.y, size.z, 1e-6)
      scale = (scene_scale * size_fraction) / max_dim
      self.mesh.setScale(scale)
      self.mesh.setPos(-center * scale)
    self.mesh_pivot.setPos(cv_to_panda(offset_cv))
    self.mesh_pivot.setHpr(*hpr)

  def place_axes(self, axis_extents, tick_spacings):
    """Draws a numbered RGB=XYZ axes gizmo in the current basis (children of
    basis_node), reaching `axis_extents[name] = (neg_len, pos_len)` in each
    direction, with tick labels spaced every `tick_spacings[name]` units."""
    if self.axes_node is not None:
      self.axes_node.removeNode()
    self.axes_node = self.basis_node.attachNewNode("axes")
    axis_specs = (('X', (1, 0, 0), (1, 0, 0, 1)),
                  ('Y', (0, 1, 0), (0, 1, 0, 1)),
                  ('Z', (0, 0, 1), (0, 0, 1, 1)))

    def add_label(text, local_pos, color, scale):
      label = TextNode(f'axis_label_{text}_{local_pos}')
      label.setText(text)
      label.setTextColor(*color)
      label.setAlign(TextNode.ACenter)
      label_np = self.axes_node.attachNewNode(label)
      label_np.setPos(local_pos)
      label_np.setScale(scale)
      label_np.setBillboardPointEye()

    segs = LineSegs()
    segs.setThickness(3)
    for name, direction, color in axis_specs:
      neg_len, pos_len = axis_extents[name]
      segs.setColor(*color)
      segs.moveTo(cv_to_panda(tuple(-d * neg_len for d in direction)))
      segs.drawTo(cv_to_panda(tuple(d * pos_len for d in direction)))
    self.axes_node.attachNewNode(segs.create())

    for name, direction, color in axis_specs:
      neg_len, pos_len = axis_extents[name]
      tick_spacing = tick_spacings[name]
      tick_scale = tick_spacing * 0.35
      n_min = -int(neg_len // tick_spacing)
      n_max = int(pos_len // tick_spacing)
      for n in range(n_min, n_max + 1):
        if n == 0:
          continue
        value = n * tick_spacing
        tick_pos = cv_to_panda(tuple(d * value for d in direction))
        add_label(f'{value:g}', tick_pos, color, tick_scale)
      tip_pos = cv_to_panda(tuple(d * pos_len for d in direction))
      add_label(name, tip_pos, color, tick_scale * 1.4)

    min_tick_scale = min(tick_spacings.values()) * 0.35
    add_label('0', Vec3(0, 0, 0), (1, 1, 1, 1), min_tick_scale * 1.2)

  def hide_axes(self):
    if self.axes_node is not None:
      self.axes_node.removeNode()
      self.axes_node = None

  def set_camera_pose(self, position_cv, forward_cv, up_cv):
    pos = cv_to_panda(position_cv)
    fwd = cv_to_panda(forward_cv)
    up = cv_to_panda(up_cv)
    self.camera.setPos(Point3(*pos))
    self.camera.lookAt(Point3(*(pos + fwd)), up)

  def set_lens_from_intrinsics(self, focal_length_px, image_width, image_height):
    self.camLens.setFilmSize(image_width, image_height)
    self.camLens.setFocalLength(focal_length_px)
    self.camLens.setAspectRatio(image_width / image_height)

  def render_to_image(self):
    self.graphicsEngine.renderFrame()
    self.graphicsEngine.renderFrame()

    tex = self.win.getScreenshot()
    width, height = tex.getXSize(), tex.getYSize()
    data = tex.getRamImageAs("RGBA")
    array = np.frombuffer(data, dtype=np.uint8).reshape((height, width, 4))
    array = np.flipud(array)  # Panda's origin is bottom-left; image formats are top-left
    return Image.fromarray(array, "RGBA")


In [ ]:
# @title Place axes: load camera track
frame_paths = sorted(rgb_raw_dir.glob('*.png'))
if not frame_paths:
  raise RuntimeError(f'No frames found in {rgb_raw_dir}.')

camera_track_path = root_dir / 'camera_track.json'
with camera_track_path.open() as f:
  track = json.load(f)
if len(track) != len(frame_paths):
  raise RuntimeError(
      f'camera_track.json has {len(track)} entries but {rgb_raw_dir} has '
      f'{len(frame_paths)} frames -- re-run "Export camera track" above.')

first_cam = track[0]
first_frame_bgr = cv2.imread(str(frame_paths[0]))
frame_h, frame_w = first_frame_bgr.shape[:2]

cam_pos = np.array(first_cam['position'], dtype=np.float64)
cam_forward_raw = np.array(first_cam['forward'], dtype=np.float64)
cam_up_raw = np.array(first_cam['up'], dtype=np.float64)
print(f'Raw forward: {cam_forward_raw} (length {np.linalg.norm(cam_forward_raw):.3f})')
print(f'Raw up: {cam_up_raw} (length {np.linalg.norm(cam_up_raw):.3f})')


In [ ]:
# @title Compute camera basis
def compute_camera_basis(forward_raw, up_raw):
  """Normalizes a camera's forward/up direction vectors to unit length, and
  derives right (perpendicular to both) and down (opposite of up)."""
  forward = forward_raw / np.linalg.norm(forward_raw)
  up = up_raw / np.linalg.norm(up_raw)
  right = np.cross(forward, up)
  right = right / np.linalg.norm(right)
  down = -up
  return forward, up, right, down


In [ ]:
# @title Compute projection helpers
cam_forward, cam_up, cam_right, cam_down = compute_camera_basis(cam_forward_raw, cam_up_raw)

focal_px = first_cam.get('focal_px', frame_w)
principal_point = (frame_w / 2.0, frame_h / 2.0)

track_positions = np.array([t['position'] for t in track])
scene_depth = float(np.median(np.linalg.norm(track_positions - cam_pos, axis=1)))


def project(point_cv):
  """Pinhole-projects a world point (CV convention) using the first frame's
  camera; returns None if the point is behind the camera."""
  point = np.asarray(point_cv, dtype=np.float64)
  local = point - cam_pos
  x = np.dot(local, cam_right)
  y = np.dot(local, cam_down)
  z = np.dot(local, cam_forward)
  if z <= 1e-6:
    return None
  return np.array([focal_px * x / z + principal_point[0],
                    focal_px * y / z + principal_point[1]])


def in_frame(point_cv, margin=0.04):
  pixel = project(point_cv)
  if pixel is None:
    return False
  mx, my = margin * frame_w, margin * frame_h
  return (mx <= pixel[0] <= frame_w - mx) and (my <= pixel[1] <= frame_h - my)


def solve_extent(origin_cv, direction, max_t):
  """Bisects how far a ray from origin_cv along `direction` can travel before
  its projection leaves the frame (with a small margin)."""
  origin = np.asarray(origin_cv, dtype=np.float64)
  direction = np.asarray(direction, dtype=np.float64)
  if not in_frame(origin):
    return 0.0
  if in_frame(origin + direction * max_t):
    return max_t
  lo, hi = 0.0, max_t
  for _ in range(30):
    mid = (lo + hi) / 2
    if in_frame(origin + direction * mid):
      lo = mid
    else:
      hi = mid
  return lo


def nice_round(value):
  """Rounds up to the nearest 1/2/5 x 10^n, for readable tick spacing."""
  if value <= 0:
    return 1.0
  exponent = np.floor(np.log10(value))
  fraction = value / (10 ** exponent)
  nice = 1 if fraction <= 1 else 2 if fraction <= 2 else 5 if fraction <= 5 else 10
  return nice * (10 ** exponent)


axes_max_t = scene_depth * 3
axes_tick_spacing = 0  # @param {type: 'number'}

if 'mesh_renderer' not in globals():
  mesh_renderer = MeshRenderer(None, (frame_w, frame_h))

In [ ]:
# @title Composite: define table-plane basis.
# @markdown Click 3 points on the table's surface in the image below (e.g.
# @markdown three corners, or any 3 non-collinear spots). Each click is
# @markdown snapped to the nearest point in the SfM sparse point cloud whose
# @markdown projection lands closest to that pixel in the first frame, giving
# @markdown each click a real, triangulated 3D position -- the plane through
# @markdown those 3 points is then solved directly, no manual tilt-tuning
# @markdown needed. Re-run this cell if the clicks land somewhere bad (check
# @markdown the per-click snap distance printed below -- a large one usually
# @markdown means that spot is short on reconstructed points, e.g. a
# @markdown texture-less patch of the table).
# @markdown
# @markdown On Colab, an interactive click prompt opens below (once per
# @markdown point) and overwrites the `table_pixel_*` fields with wherever you
# @markdown click; outside Colab (or if the click prompt fails) it falls back
# @markdown to whatever pixel coordinates are typed into those six fields.

table_pixel_1_x = frame_w // 4  # @param {type: 'integer'}
table_pixel_1_y = frame_h * 3 // 4  # @param {type: 'integer'}
table_pixel_2_x = frame_w * 3 // 4  # @param {type: 'integer'}
table_pixel_2_y = frame_h * 3 // 4  # @param {type: 'integer'}
table_pixel_3_x = frame_w // 2  # @param {type: 'integer'}
table_pixel_3_y = frame_h // 2  # @param {type: 'integer'}

clicked_pixels = [(table_pixel_1_x, table_pixel_1_y),
                   (table_pixel_2_x, table_pixel_2_y),
                   (table_pixel_3_x, table_pixel_3_y)]

try:
  import base64
  import io
  from google.colab.output import eval_js
  from IPython.display import Javascript

  buffered = io.BytesIO()
  Image.fromarray(cv2.cvtColor(first_frame_bgr, cv2.COLOR_BGR2RGB)).save(buffered, format='PNG')
  image_b64 = base64.b64encode(buffered.getvalue()).decode()

  def capture_click(prompt_text):
    """Draws the frame in an <img>, waits for one click, converts the
    click's on-screen coordinates back to original-image pixel coordinates
    (the image is CSS-scaled to fit, so this can't just use event offsets
    directly), then removes the element and returns [x, y]."""
    click_js = f'''
    async function getTableClick() {{
      const container = document.createElement('div');
      const hint = document.createElement('div');
      hint.textContent = {json.dumps(prompt_text)};
      const img = document.createElement('img');
      img.src = 'data:image/png;base64,{image_b64}';
      img.style.cursor = 'crosshair';
      img.style.maxWidth = '100%';
      container.appendChild(hint);
      container.appendChild(img);
      document.body.appendChild(container);
      await new Promise(resolve => {{ img.onload = resolve; }});
      return new Promise(resolve => {{
        img.onclick = (event) => {{
          const rect = img.getBoundingClientRect();
          const x = Math.round((event.clientX - rect.left) * (img.naturalWidth / rect.width));
          const y = Math.round((event.clientY - rect.top) * (img.naturalHeight / rect.height));
          container.remove();
          resolve([x, y]);
        }};
      }});
    }}
    getTableClick()
    '''
    return eval_js(click_js)

  clicked_pixels = []
  for i in range(3):
    clicked = capture_click(f'Click table point {i + 1} of 3...')
    clicked_pixels.append((int(clicked[0]), int(clicked[1])))
    print(f'Clicked point {i + 1}: {clicked_pixels[-1]}')
except ImportError:
  print('Not running on Colab (or click capture unavailable) -- using '
        f'table_pixel_* fields above: {clicked_pixels}')


def project_all(points_cv):
  """Batch version of project() (defined above) for the whole sparse cloud."""
  local = np.asarray(points_cv, dtype=np.float64) - cam_pos
  x = local @ cam_right
  y = local @ cam_down
  z = local @ cam_forward
  in_front = z > 1e-6
  z_safe = np.where(in_front, z, 1.0)
  pixels = np.stack([focal_px * x / z_safe + principal_point[0],
                      focal_px * y / z_safe + principal_point[1]], axis=-1)
  return pixels, in_front


sparse_pixels, sparse_in_front = project_all(sparse_points_cv)


def nearest_sparse_point(pixel_xy):
  dists = np.linalg.norm(sparse_pixels - np.asarray(pixel_xy), axis=1)
  dists = np.where(sparse_in_front, dists, np.inf)
  idx = np.argmin(dists)
  return sparse_points_cv[idx], dists[idx]


table_points_cv = []
for i, pixel in enumerate(clicked_pixels):
  point, dist_px = nearest_sparse_point(pixel)
  table_points_cv.append(point)
  print(f'Point {i + 1}: pixel {pixel} -> sparse point {point.tolist()} '
        f'({dist_px:.1f}px from click)')
p1, p2, p3 = table_points_cv

edge1 = p2 - p1
edge2 = p3 - p1
table_normal_cv = np.cross(edge1, edge2)
norm = np.linalg.norm(table_normal_cv)
if norm < 1e-9:
  raise RuntimeError(
      'The 3 clicked points are (nearly) collinear -- pick 3 points that '
      "aren't all on one line.")
table_normal_cv /= norm

table_origin_cv = tuple(np.mean(table_points_cv, axis=0).tolist())
# Orient "up" toward the camera's side of the plane, since the shot is
# presumably looking down at the table from above it, not from underneath.
if np.dot(table_normal_cv, cam_pos - np.asarray(table_origin_cv)) < 0:
  table_normal_cv = -table_normal_cv
table_up_cv = tuple(table_normal_cv.tolist())

table_right_cv = edge1 / np.linalg.norm(edge1)
table_forward_cv = np.cross(table_normal_cv, table_right_cv)
table_right_cv = tuple(table_right_cv.tolist())
table_forward_cv = tuple(table_forward_cv.tolist())

table_axis_dirs = {'X': np.array(table_right_cv), 'Y': -np.array(table_up_cv),
                    'Z': np.array(table_forward_cv)}
table_axis_extents = {}
table_axis_tick_spacings = {}
for name, direction in table_axis_dirs.items():
  neg_len = solve_extent(table_origin_cv, -direction, axes_max_t)
  pos_len = solve_extent(table_origin_cv, direction, axes_max_t)
  table_axis_extents[name] = (neg_len, pos_len)
  table_axis_tick_spacings[name] = (
      nice_round((neg_len + pos_len) / 5) if axes_tick_spacing <= 0 else axes_tick_spacing)

mesh_renderer.mesh_pivot.hide()
mesh_renderer.set_basis(table_origin_cv, table_right_cv, table_forward_cv, table_up_cv)
mesh_renderer.place_axes(table_axis_extents, table_axis_tick_spacings)
mesh_renderer.set_camera_pose(first_cam['position'], first_cam['forward'], first_cam['up'])
if 'focal_px' in first_cam:
  mesh_renderer.set_lens_from_intrinsics(first_cam['focal_px'], frame_w, frame_h)
table_axes_rgba = mesh_renderer.render_to_image().resize((frame_w, frame_h))
first_frame_rgba = Image.fromarray(cv2.cvtColor(first_frame_bgr, cv2.COLOR_BGR2RGB)).convert('RGBA')
table_frame = Image.alpha_composite(first_frame_rgba, table_axes_rgba).convert('RGB')
display(table_frame)
mesh_renderer.hide_axes()


In [ ]:
# @title Composite: calibrate mesh placement relative to the table plane.
# @markdown `mesh_offset_cv` and `mesh_hpr` are both relative to the
# @markdown table-plane basis defined above (X=right, Y=down, Z=forward,
# @markdown with the plane itself spanning X/Y and Z pointing up out of the
# @markdown table) -- e.g. offset (0, 0, 0) sits right where you clicked, and
# @markdown mesh_hpr controls heading/pitch/roll about that same point. Re-run
# @markdown this cell after any change; it composites onto the axes frame
# @markdown baked above so you can check the placement against the table.

mesh_offset_cv = (-1,0,3)  # @param {type: 'raw'}
mesh_hpr = (45.0, 0, 0.0)  # @param {type: 'raw'}
model_path = None  # @param {type: 'raw'}
mesh_size_fraction = 0.3  # @param {type: 'number'}

# @markdown Check this to use the downloaded animated mutant.glb instead of
# @markdown `model_path` above -- see "Render tracked mesh onto video: setup"
# @markdown below to actually play its animation (set `anim_clip_name` to
# @markdown one of its clips, e.g. 'mixamo.com').
use_mutant_model = True  # @param {type: 'boolean'}
if use_mutant_model:
  model_path = str(mutant_model_path)

mesh_world_position_cv = basis_point_to_world_cv(
    mesh_offset_cv, table_origin_cv, table_right_cv, table_forward_cv, table_up_cv)
scene_scale = float(np.median(
    np.linalg.norm(track_positions - mesh_world_position_cv, axis=1)))

mesh_renderer.reset_mesh(model_path)
mesh_renderer.set_basis(table_origin_cv, table_right_cv, table_forward_cv, table_up_cv)
mesh_renderer.place_mesh(mesh_offset_cv, mesh_size_fraction, scene_scale, hpr=mesh_hpr)
mesh_renderer.set_camera_pose(first_cam['position'], first_cam['forward'], first_cam['up'])
if 'focal_px' in first_cam:
  mesh_renderer.set_lens_from_intrinsics(first_cam['focal_px'], frame_w, frame_h)
mesh_rgba = mesh_renderer.render_to_image().resize((frame_w, frame_h))
calibration_preview = Image.alpha_composite(table_frame.convert('RGBA'), mesh_rgba).convert('RGB')
display(calibration_preview)

calibration_path = root_dir / 'mesh_calibration.json'
with calibration_path.open('w') as f:
  json.dump({
      'basis_origin_cv': list(table_origin_cv),
      'basis_right_cv': list(table_right_cv),
      'basis_forward_cv': list(table_forward_cv),
      'basis_up_cv': list(table_up_cv),
      'mesh_offset_cv': list(mesh_offset_cv),
      'mesh_hpr': list(mesh_hpr),
      'mesh_size_fraction': mesh_size_fraction,
      'model_path': model_path,
  }, f, indent=2)
print(f'Wrote calibration to {calibration_path}')


In [ ]:
# @title Render tracked mesh onto video: setup
calibration_path = root_dir / 'mesh_calibration.json'
anchor_point_path = root_dir / 'anchor_point.json'


def use_identity_basis():
  return (0.0, 0.0, 0.0), (1.0, 0.0, 0.0), (0.0, 0.0, 1.0), (0.0, -1.0, 0.0)


if calibration_path.exists():
  with calibration_path.open() as f:
    calibration = json.load(f)
  basis_origin_cv = tuple(calibration['basis_origin_cv'])
  basis_right_cv = tuple(calibration['basis_right_cv'])
  basis_forward_cv = tuple(calibration['basis_forward_cv'])
  basis_up_cv = tuple(calibration['basis_up_cv'])
  mesh_offset_cv = tuple(calibration['mesh_offset_cv'])
  mesh_hpr = tuple(calibration['mesh_hpr'])
  mesh_size_fraction = calibration['mesh_size_fraction']
  model_path = calibration['model_path']
elif anchor_point_path.exists():
  with anchor_point_path.open() as f:
    mesh_offset_cv = tuple(json.load(f)['anchor_position_cv'])
  basis_origin_cv, basis_right_cv, basis_forward_cv, basis_up_cv = use_identity_basis()
  mesh_hpr = (0.0, 0.0, 0.0)
  mesh_size_fraction = 0.35
  model_path = None
else:
  raise RuntimeError(
      'No mesh_calibration.json or anchor_point.json found -- run the '
      'calibration cells above at least once before this one.')

frame_paths = sorted(rgb_raw_dir.glob('*.png'))
if not frame_paths:
  raise RuntimeError(f'No frames found in {rgb_raw_dir}.')
camera_track_path = root_dir / 'camera_track.json'
with camera_track_path.open() as f:
  track = json.load(f)
if len(track) != len(frame_paths):
  raise RuntimeError(
      f'camera_track.json has {len(track)} entries but {rgb_raw_dir} has '
      f'{len(frame_paths)} frames -- re-run "Export camera track" above.')

first_frame_bgr = cv2.imread(str(frame_paths[0]))
frame_h, frame_w = first_frame_bgr.shape[:2]

if 'mesh_renderer' not in globals():
  mesh_renderer = MeshRenderer(model_path, (frame_w, frame_h))
else:
  mesh_renderer.reset_mesh(model_path)
mesh_renderer.hide_axes()
mesh_renderer.set_basis(basis_origin_cv, basis_right_cv, basis_forward_cv, basis_up_cv)

# @markdown The mutant model's rig only has one animation clip worth using
# @markdown ("mixamo.com.001" -- the other, "mixamo.com", is a short static
# @markdown T-pose-ish clip). Hardcoded here rather than a free-text field,
# @markdown since it only applies when use_mutant_model is on above; any
# @markdown other model (or the built-in Panda3D placeholder, used when no
# @markdown model_path is provided) has no matching clip and renders in its
# @markdown rest pose.
anim_clip_name = 'mixamo.com.001' if use_mutant_model else ''
available_anim_clips = mesh_renderer.mesh.getAnimNames()
if anim_clip_name and anim_clip_name in available_anim_clips:
  print(f'Will animate the mesh using clip {anim_clip_name!r}.')
elif anim_clip_name:
  print(f'WARNING: anim_clip_name={anim_clip_name!r} does not match any '
        f'available clip {available_anim_clips} -- the mesh will render '
        'in its rest pose (no animation).')
else:
  print('No animation clip selected -- the mesh will render in its rest pose.')

mesh_world_position_cv = basis_point_to_world_cv(
    mesh_offset_cv, basis_origin_cv, basis_right_cv, basis_forward_cv, basis_up_cv)
track_positions = np.array([t['position'] for t in track])
scene_scale = float(np.median(np.linalg.norm(track_positions - mesh_world_position_cv, axis=1)))
mesh_renderer.place_mesh(mesh_offset_cv, mesh_size_fraction, scene_scale, hpr=mesh_hpr)

# rgb_raw_dir holds every native frame of the source video, and
# camera_track.json has one entry per native frame -- so encoding the output
# at the video's own native frame rate reproduces the source video exactly
# (same frame count, same duration), no fps math needed.
output_fps = input_fps

import subprocess

output_video_path = root_dir / f'{capture_name}_composited.mp4'
# cv2.VideoWriter's 'mp4v' fourcc encodes old, poorly-supported MPEG-4 Part 2
# (many players -- Windows Media Player, QuickTime, some mobile/browser
# players -- can't decode it even though ffplay/VLC usually can). Pipe raw
# frames to ffmpeg instead and encode proper H.264/yuv420p, which is
# universally playable.
ffmpeg_proc = subprocess.Popen(
    ['ffmpeg', '-y',
     '-f', 'rawvideo', '-vcodec', 'rawvideo',
     '-pix_fmt', 'bgr24', '-s', f'{frame_w}x{frame_h}', '-r', str(output_fps),
     '-i', '-',
     '-an', '-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
     str(output_video_path)],
    stdin=subprocess.PIPE)


In [ ]:
# @title Define composite_onto_frame
def composite_onto_frame(frame_path, mesh_rgba):
  """Reads one real video frame from disk and alpha-composites the rendered
  mesh on top of it. Returns a BGR uint8 array ready for the video encoder."""
  frame_bgr = cv2.imread(str(frame_path))
  frame_rgba = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)).convert('RGBA')
  composited = Image.alpha_composite(frame_rgba, mesh_rgba)
  return cv2.cvtColor(np.array(composited.convert('RGB')), cv2.COLOR_RGB2BGR)

In [ ]:
# @title Render tracked mesh onto video: run
for frame_idx, (frame_path, t) in enumerate(zip(frame_paths, track)):
  mesh_renderer.set_camera_pose(t['position'], t['forward'], t['up'])
  if 'focal_px' in t:
    mesh_renderer.set_lens_from_intrinsics(t['focal_px'], frame_w, frame_h)
  # If the mesh has an embedded animation and anim_clip_name was set above,
  # advance that clip to match this frame's timestamp (looping for the
  # length of the video) before rendering it.
  if anim_clip_name:
    mesh_renderer.set_anim_time(anim_clip_name, frame_idx / input_fps)
  mesh_rgba = mesh_renderer.render_to_image().resize((frame_w, frame_h))

  out_bgr = composite_onto_frame(frame_path, mesh_rgba)
  ffmpeg_proc.stdin.write(out_bgr.tobytes())

  if (frame_idx + 1) % 50 == 0:
    print(f'  composited {frame_idx + 1} frames')

ffmpeg_proc.stdin.close()
ffmpeg_proc.wait()
if ffmpeg_proc.returncode != 0:
  raise RuntimeError(f'ffmpeg exited with code {ffmpeg_proc.returncode}')
print(f'Wrote {len(frame_paths)} composited frames to {output_video_path}')


In [ ]:
# @title Composite: preview and download.
from base64 import b64encode
from IPython.display import HTML

mp4_bytes = output_video_path.read_bytes()
data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

# try:
#   from google.colab import files
#   files.download(str(output_video_path))
# except ImportError:
#   pass  # Not running on Colab -- the file is already on local disk at output_video_path.